# 01 - Extração Distribuída no Cluster Spark (ADLS Gen2)
**Squad 2 — Real Time for Business | Dupla 1**  
**Integrantes:** Lucas Sousa Santos Oliveira & Zaiden Emiliano Segundo Seleme  
**Tabelas:** `ecommerce_produtos` e `ecommerce_categorias`  
**Origem:** `abfss://raw@internshipdatalake.dfs.core.windows.net/real-time-data/`  

### Objetivo Arquitetural (Task 2):
1. **Processamento 100% Distribuído:** Realizar a leitura paralelizada diretamente nos executores do cluster Spark através do protocolo `abfss://`, sem trafegar dados no nó driver (evitando o gargalo de memória única).
2. **Autenticação Granular por Chamada:** Passar as credenciais OAuth do Service Principal via `.options(...)` em cada chamada `spark.read`, contornando a restrição de injeção global de configurações no Databricks Serverless / Free Edition.
3. **Consolidação e Deduplicação:** Ler recursivamente todas as partições temporais (`YYYY/MM/DD/HHMMSS/*.parquet`) e aplicar a deduplicação distribuída por chave primária.

In [0]:
# Carregamento seguro das variáveis de ambiente
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override=True)

storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

print("Ambiente configurado:")
print(f"  Storage Account: {storage_account}")
print(f"  Client ID carregado: {client_id is not None}")
print(f"  Tenant ID carregado: {tenant_id is not None}")

## 1. Dicionário de Opções OAuth para o Conector Spark ABFS

In [0]:
# Configurações OAuth do Service Principal passadas a cada chamada do Spark
# Isso garante que a autenticação ocorra diretamente nos executores do cluster
adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

print("Dicionário adls_options preparado para injeção no spark.read.")

## 2. Leitura Paralelizada e Distribuída de `ecommerce_produtos`

In [0]:
# Leitura distribuída de todas as janelas temporais de produtos
# O Spark lê em paralelo todos os arquivos Parquet que correspondem ao padrão
caminho_produtos = f"abfss://raw@{storage_account}.dfs.core.windows.net/real-time-data/*/*/*/*/ecommerce_produtos.parquet"

print(f"Executando spark.read distribuído em: {caminho_produtos}")
df_produtos_raw = spark.read.options(**adls_options).parquet(caminho_produtos)

total_bruto_prod = df_produtos_raw.count()
print(f"Total de registros brutos lidos (distribuído): {total_bruto_prod}")
print("Estrutura do DataFrame:")
df_produtos_raw.printSchema()

## 3. Leitura Paralelizada e Distribuída de `ecommerce_categorias`

In [0]:
# Leitura distribuída de todas as janelas temporais de categorias
caminho_categorias = f"abfss://raw@{storage_account}.dfs.core.windows.net/real-time-data/*/*/*/*/ecommerce_categorias.parquet"

print(f"Executando spark.read distribuído em: {caminho_categorias}")
df_categorias_raw = spark.read.options(**adls_options).parquet(caminho_categorias)

total_bruto_cat = df_categorias_raw.count()
print(f"Total de registros brutos lidos (distribuído): {total_bruto_cat}")
print("Estrutura do DataFrame:")
df_categorias_raw.printSchema()

## 4. Deduplicação Distribuída por Chave Primária

In [0]:
# Deduplicação realizada pelo cluster Spark
# Produtos: chave primária 'sku'
df_produtos = df_produtos_raw.dropDuplicates(["sku"])
total_unicos_prod = df_produtos.count()

# Categorias: chave primária 'id_categoria'
df_categorias = df_categorias_raw.dropDuplicates(["id_categoria"])
total_unicos_cat = df_categorias.count()

print("=== Consolidação Final de Tempo Real (Distribuído) ===")
print(f"Produtos:   {total_bruto_prod} linhas brutas -> {total_unicos_prod} SKUs únicos")
print(f"Categorias: {total_bruto_cat} linhas brutas -> {total_unicos_cat} Categorias únicas")

print("\nAmostragem de Produtos (5 registros):")
display(df_produtos.limit(5))

print("\nAmostragem de Categorias (5 registros):")
display(df_categorias.limit(5))